# Session 13：需要時才打開的練習素材庫

這不是今天一定要跑完的課程。先和導師談談最近的作業，以及你希望得到什麼幫助，再挑一個用得上的例子。

本檔使用合成資料，不含真實學生或商店資料。每一行代表一家門店的一年；sales 和 profit 的單位都設定為萬元人民幣。

## 如何挑選

- 只想看懂表格：模組 0 → 1。
- 想檢查資料：模組 0 → 1 → 2，三種檢查可分開看。
- 想學篩選：模組 0 → 1 → 4。
- 想學分組加總：模組 0 → 1 → 5，不必先做篩選。
- 想畫五年合計圖：模組 0 → 1 → 5 → 6A → 6B。
- 想做 2025 比較或趨勢題：模組 0 → 1 → 6A，再選 7 或 8。
- 模組 3 是選讀。只有想分清 Series 和 DataFrame 時才看。

可以跳過主題，但仍需先執行列出的前置格。後面的格子會使用前面建立的變數，Notebook 不會自動替你補跑。

每段都分成「先做什麼」「應看到什麼」「為什麼」「如何確認」。不必每一步都猜結果；先弄清楚現在想回答的問題就好。

## 模組 0：確認現在使用的 Python 與檔案位置

**先做什麼：** 選好 Python kernel，執行下面這一格。請先下載完整 session13 資料夾，保留其中的 data 子目錄。

**應看到什麼：** Python 的執行路徑、pandas 與 matplotlib 的版本，以及 CSV 和輸出資料夾的位置。

**為什麼：** Notebook 使用的 Python 不一定和終端機相同。我們先確認這個 Python 能找到需要的套件和檔案。

**如何確認：** 沒有紅色報錯，且 CSV 路徑指向本課的 sample-sales.csv。這只代表環境格成功，不代表後面的分析已經完成。

如果看到「No module named pandas」，表示目前這個 Python 找不到 pandas。請先和導師確認環境，再處理安裝；不用反覆重新建立 notebook。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python 路徑：", sys.executable)
print("pandas 版本：", pd.__version__)
print("matplotlib 版本：", matplotlib.__version__)
print("目前工作目錄：", Path.cwd())

# 從 session13 資料夾或倉庫根目錄開啟，都可以找到同一份 CSV。
lesson_dir = Path.cwd()
if not (lesson_dir / "data/sample-sales.csv").is_file():
    lesson_dir = lesson_dir / "session13"
csv_path = lesson_dir / "data/sample-sales.csv"
if not csv_path.is_file():
    raise FileNotFoundError("找不到 CSV。請從 session13 或倉庫根目錄開啟，並確認 data 資料夾已下載。")
output_dir = lesson_dir / "outputs"
output_dir.mkdir(exist_ok=True)
print("CSV 位置：", csv_path.resolve())
print("圖片輸出位置：", output_dir.resolve())

## 模組 1：把 CSV 讀成一張表

前置：模組 0。

**先做什麼：** 執行下面一格。

**應看到什麼：** 第一段輸出是 `(20, 5)`，表示整張表有 20 行、5 欄。接下來只顯示前 5 行，方便先看資料長相。

**為什麼：** `df.shape` 查看整張表的大小；`df.head()` 預覽前 5 行，沒有刪除其餘 15 行。

**如何確認：** 預覽中的前 5 行都是門店 A，年份從 2021 到 2025。完整資料仍有 A、B、C、D 四家店。

`pd` 是我們替 pandas 取的簡稱。`df` 是用來存這張表的變數名，並不是 Python 規定的名字。也可以取名 sales_table，但後面使用它的地方就要一致改名。先不需要為此改代碼。

In [ ]:
df = pd.read_csv(csv_path)
print("整張表的行數、欄數：", df.shape)
print("下面只預覽前 5 行：")
print(df.head().to_string(index=False))

### 先讀懂一行，再繼續分析

第一行表示：門店 A 在 2021 年的銷售額是 120 萬元，利潤是 18 萬元。

category 欄的「電子」只是該店當年的主力品類標記。它沒有告訴我們電子商品單獨賺了多少，因此不能把這行的 18 萬元直接解讀為「電子類利潤」。

如果現在需要的只是理解資料，到這裡就可以回到自己的問題。

## 模組 2：檢查資料，三份報告分開看

前置：模組 0、1。只有需要檢查資料時才做，不用一次讀完三份報告。

### 2A：每個欄位目前被當成哪一種資料？

**先做什麼：** 執行下一格。

**應看到什麼：** year、sales、profit 通常顯示 int64，意思是整數；store 和 category 在不同 pandas 版本可能顯示 object 或 str，這份樣本的內容是文字。

**為什麼：** dtype 是「資料型別」的簡稱。數字若被讀成文字，後面計算可能不符合預期。object 本身也不保證每個值都是文字，仍要看內容。

**如何確認：** 年份、銷售額、利潤是可以計算的數字。這份報告不會同時告訴你是否有缺值或重複，下面是另外兩種檢查。

In [ ]:
print(df.dtypes)

### 2B：有多少格沒有資料？

**先做什麼：** 執行下一格。

**應看到什麼：** 每個欄位後面都是 0，表示本樣本沒有 pandas 判定為缺值的格子。

**為什麼：** `isna()` 找缺值，再用 `sum()` 計算每欄有幾個。這裡加的是缺值標記，不是利潤。

**如何確認：** 看完整報告，不只看 head 的前 5 行。真實資料若出現缺值，先問它代表什麼，不直接填成 0。空白字串或其他特殊標記也可能需要另外確認。

In [ ]:
print(df.isna().sum())

### 2C：同一家店、同一年是否出現不只一行？

**先做什麼：** 執行下一格。

**應看到什麼：** 0，表示本樣本沒有重複的店年組合。

**為什麼：** 這份資料約定一店一年只有一行，所以把 store 與 year 合在一起檢查。若真實資料的一行是訂單，同店同年有多行反而很正常。

**如何確認：** 先知道一行代表什麼，再判斷重複是不是問題。不要只因為看到重複就刪除。

In [ ]:
print(df.duplicated(["store", "year"]).sum())

## 模組 3：只取一欄（選讀）

前置：模組 0、1。這段只在你想知道兩種括號差別時使用，不影響你先學篩選或分組。

**先做什麼：** 執行下一格。

**應看到什麼：** 第一個結果是 Series，shape 為 `(20,)`；第二個是 DataFrame，shape 為 `(20, 1)`。

**為什麼：** Series 是一串有標籤的資料，這裡仍然有 20 個利潤值，並不是只有一個數字。DataFrame 是二維表，這裡只有一欄、20 行。兩者沒有誰比較高階。

**如何確認：** 兩個版本都保留 20 筆資料，只是資料結構不同。本 CSV 欄名是小寫 profit；Python 區分大小寫，寫成 Profit 會找不到這個欄位。

In [ ]:
profit_column = df["profit"]
profit_table = df[["profit"]]
print(type(profit_column).__name__, profit_column.shape)
print(type(profit_table).__name__, profit_table.shape)

## 模組 4：只留下利潤大於 20 萬元的行

前置：模組 0、1。不需要先執行模組 2 或 3。

**先做什麼：** 執行下一格，先看行數，再看留下來的資料。

**應看到什麼：** filtered 有 9 行、5 欄；原本的 df 仍有 20 行、5 欄。畫面下方為了方便閱讀，只列出 store、year、profit 這 3 欄。

**為什麼：** 第一句代碼篩掉不符合條件的行。最後一句另外選出 3 欄來顯示。篩行和選欄是兩個不同動作；filtered 裡仍有全部 5 欄。

**如何確認：** 門店 A 有 4 行符合，門店 C 有 5 行符合，合計 9 行。條件是大於 20，不包含等於 20；這份樣本剛好沒有利潤等於 20 的行。

In [ ]:
filtered = df[df["profit"] > 20]
print("篩選後的行數、欄數：", filtered.shape)
print("原表的行數、欄數：", df.shape)
print("下面另外只顯示 3 欄：")
print(filtered[["store", "year", "profit"]].to_string(index=False))

### 需要練一個改動時：把 20 改成 30

先問自己：「利潤剛好 30 的行，會不會留下？」

把上面條件改成 `profit > 30`，只重跑那一格。應留下 5 行：門店 A 的 2025 年，以及門店 C 的 2021、2023、2024、2025 年。剛好 30 的行不會留下。

練完若要回到本頁的標準結果，再改回 20。無論門檻怎麼改，原始 df 都不會被這段代碼覆寫。

## 模組 5：把一家店的五行資料，加成一個總數

前置：模組 0、1。這一段使用完整的 df，不使用前面的 filtered。

### 5A：先只看門店 A

**先做什麼：** 執行下面一格，讀出門店 A 的五年利潤。

**應看到什麼：** 2021 到 2025 年分別是 18、22、25、30、35 萬元。

**為什麼：** 先把會被加在一起的原始行攤開，才知道總數代表什麼。

**如何確認：** 用計算機或心算做一次 18 + 22 + 25 + 30 + 35，應得到 130 萬元。

In [ ]:
store_a_rows = df[df["store"] == "門店A"]
print(store_a_rows[["year", "profit"]].to_string(index=False))

### 5B：四家店都做同樣的加總

**先做什麼：** 執行下一格。

**應看到什麼：** 原表有 20 行，結果是 4 個總數：A = 130、B = 71、C = 193、D = 62。單位仍是萬元。

**為什麼：** `groupby("store")` 把同一家店的行放在一起，`["profit"]` 指定要算利潤，`sum()` 將每組利潤相加。每家店的五年變成一個數字，所以結果不再保留每一年的明細。

**如何確認：** 先對照剛才手算的 A = 130，再抽查 B 的 12 + 14 + 11 + 15 + 19 = 71。

注意：這回答的是「各店五年利潤總和」。如果題目要 2025 年或逐年趨勢，這張結果表就不適合直接拿去畫圖。

In [ ]:
totals = df.groupby("store")["profit"].sum()
print("原表行數：", len(df))
print("店別總數的個數：", len(totals))
print(totals)

### 5C：再做一個整體核對

**先做什麼：** 執行下一格。

**應看到什麼：** 兩個合計都是 456。

**為什麼：** 我們只是按店重新分組，沒有刻意刪掉任何行，所以四家店總數再相加，應等於原表全部利潤相加。

**如何確認：** 兩個數字一致是一項證據，但不能取代對題意的確認。例如題目其實只要 2025 年，算出相等的五年總和仍然答錯問題。

In [ ]:
print("四家店的總數再相加：", totals.sum())
print("原表全部利潤相加：", df["profit"].sum())

## 模組 6A：畫圖前的共用設定

前置：模組 0。只有需要畫圖時才跑；不要求記住字型設定代碼。

**先做什麼：** 執行下面一格。

**應看到什麼：** 找到常見中文字型就使用它；找不到就顯示「使用英文標籤」。這不是分析錯誤。

**為什麼：** 有些電腦沒有可用的中文字型。英文備援可以先讓圖保持可讀，不需要在課中下載字型。

**如何確認：** 之後畫出的 Store A–D 對應門店 A–D；Profit (10,000 CNY) 表示利潤，單位是萬元人民幣。

In [ ]:
from matplotlib import font_manager

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
cjk_font = next((name for name in ["Noto Sans CJK TC", "Noto Sans CJK SC", "Microsoft YaHei", "PingFang TC", "SimHei"] if name in available_fonts), None)
use_chinese = cjk_font is not None
if use_chinese:
    plt.rcParams["font.family"] = cjk_font
plt.rcParams["axes.unicode_minus"] = False
store_labels = {f"門店{letter}": f"Store {letter}" for letter in "ABCD"}
y_label = "利潤（萬元人民幣）" if use_chinese else "Profit (10,000 CNY)"
x_label = "門店" if use_chinese else "Store"
print("圖表字型：", cjk_font or "使用英文標籤")

## 模組 6B：把五年合計畫成條形圖

前置：模組 0、1、5B、6A。先核對 totals，確定你要回答的問題就是「各店五年總和」。

**先做什麼：** 執行下一格，然後打開輸出路徑中的 profit-total.png。

**應看到什麼：** 四根柱子，分別對應 130、71、193、62 萬元。標題寫明是 2021–2025 的合計，以及這是合成資料。

**為什麼：** 這張圖只是呈現 totals 裡的四個數字。畫圖不會替我們判斷應該算總和、平均，還是只算某一年。

**如何確認：** 圖上最高的是門店 C。標題、單位和四根柱子的數值應與結果表一致。圖片先保存，再顯示。

In [ ]:
plot_totals = totals if use_chinese else totals.rename(index=store_labels)
ax = plot_totals.plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_title("各店 2021–2025 利潤合計（合成資料）" if use_chinese else "Profit by store, 2021–2025 total (synthetic)")
ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
fig = ax.get_figure()
fig.tight_layout()
bar_path = output_dir / "profit-total.png"
fig.savefig(bar_path, dpi=150, bbox_inches="tight")
print("圖片已保存到：", bar_path.resolve())
plt.show()
plt.close(fig)

## 模組 7：只比較 2025 年（需要時才選）

前置：模組 0、1、6A。不需要先完成所有前面的主題。

**先做什麼：** 先說「我要比較 2025 年各店利潤」。在下面空白格完成篩選，先印出結果表，確認後才畫圖。需要 AI 幫忙時，一次只請它處理下一步。

**應看到什麼：** 2025 年有四家店各一行：A = 35、B = 19、C = 48、D = 17 萬元。圖應呈現這四個值，不是五年總和。

**為什麼：** 這次題目有年份條件，要先只留下 2025 年。

**如何確認：** 從原表抽查 C 店的 2025 年，應是 48。若要存圖，使用 outputs/profit-2025.png，標題注明 2025 年、單位和合成資料。

以下空白格不會自動產出答案或圖片。教師答案只是備援，不必一開始照抄。

In [ ]:
# 在這裡完成目前需要的一小步，例如先篩出 2025 年並印出結果。


## 模組 8：看五年的變化（需要時才選）

前置：模組 0、1、6A。只有目前問題與時間趨勢有關時，才做這題。

**先做什麼：** 保留每家店每一年的利潤，按年份從小到大排列。可以請 AI 協助把 year 放在行、store 放在欄、profit 放在格子中。

**應看到什麼：** 一張 5 行、4 欄的結果表；畫出來有四條線，每條線五個年份點。

**為什麼：** 要看變化就必須保留每年數字，不能使用只剩四個五年總數的 totals。

**如何確認：** B 店依序是 12、14、11、15、19，2022 到 2023 年下降，不是每年都上升。需要保存時使用 outputs/profit-trend.png。

In [ ]:
# 有需要再處理逐年趨勢，不必為了跑完整份素材庫而加做。


## 本次收尾：只記錄今天真正用到的部分

請用自己的話填兩三句，不需要回答整份素材庫的所有題目。

- 我今天想解決的問題是：……
- 我做了什麼，現在得到什麼結果：……
- 我怎麼核對這個結果：……
- 還需要幫忙的地方：……

保存後，從乾淨 kernel 重跑本次使用的模組及其前置步驟。完整成品可以另存精簡 notebook，只保留這次需要的內容。

真實作業、私人資料與輸出不要直接提交到公開 repo。合成資料只用於練習，不能拿來解釋真實商店為什麼賺錢。